# Answer-aware evidence selection (Colab)

This follow-up reuses the completed BM25 top-100 and question-guided train/dev checkpoints. It asks whether **extracting an answer from a source passage**, then selecting up to three cited passages, helps more than ranking by topic similarity. It does not run full-corpus dense retrieval or regenerate Qwen questions.

A compact SQuAD2 extractive QA reader scores at most 20 candidate passages per claim and saves exact source spans. A small logistic selector learns from **train-only silver answer-to-passage labels**. Its train rankings are five-fold out-of-fold; the final selector ranks dev. We then train DistilBERT twice on matched train/dev evidence: original passages versus short question/answer/source excerpts. The audit is train-only and blind to system identity. All long reader and verifier stages checkpoint to Drive; rerun the same cell after interruption.

Important: silver labels and URL matches are not human passage-relevance judgments. The answer reader can quote an irrelevant span. Inspect the audit before treating any increase as genuine evidence improvement. Do not tune this pipeline on dev labels.


In [ ]:
import os, sys, subprocess, runpy, gc, json, time
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/muyishen2040/10718-ML-In-Practice.git'
REPO_DIR = Path('/content/10718-ML-In-Practice-answer-aware')
if REPO_DIR.exists():
    dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip()
    if dirty: raise RuntimeError('Colab checkout has local edits; save them before pulling.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path: sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a Colab GPU runtime (T4 or better)'
print('GPU:', torch.cuda.get_device_name(0))
print('Repo revision:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())
for name in ('scripts/run_answer_reader.py', 'scripts/train_answer_selector.py', 'scripts/train_transformer_verifier.py', 'src/retrieval/answer_aware.py'):
    assert (REPO_DIR / name).is_file(), f'Missing {name}; pull the pushed commit and rerun setup'


In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
PROCESSED = DATA_ROOT / 'processed/averitec'
TOP100 = DATA_ROOT / 'outputs/initial_ml_solution_answer_aligned_top100_v1/bm25'
TRAIN100 = TOP100 / 'train_bm25_top100_v2/train_rankings.jsonl'
DEV100 = TOP100 / 'dev_bm25_top100_v2/dev_rankings.jsonl'
QG = DATA_ROOT / 'outputs/initial_ml_solution_question_guided_v1/retrieval'
QG_TRAIN = QG / 'train_qwen_ce32_v1/train_question_guided.partial.jsonl'
QG_DEV = QG / 'dev_qwen_ce32_v1/dev_question_guided.partial.jsonl'
ALIGNMENT = DATA_ROOT / 'outputs/initial_ml_solution_answer_aligned_top100_v1/alignment/gold_answer_top100_v1/train_alignment.jsonl'
DEV_QRELS = PROCESSED / 'dev_qrels.jsonl'
ROOT = DATA_ROOT / 'outputs/initial_ml_solution_answer_aware_v1'
TRAIN_READER = ROOT / 'reader/train_qa20_v1/train_answer_features.jsonl'
DEV_READER = ROOT / 'reader/dev_qa20_v1/dev_answer_features.jsonl'
SELECTOR = ROOT / 'selector/qa20_silver_oof_v1'
for path in (TRAIN100, DEV100, QG_TRAIN, QG_DEV, ALIGNMENT, DEV_QRELS):
    if not path.is_file(): raise FileNotFoundError(path)
for path in (QG / 'train_qwen_ce32_v1/train_metrics.json', QG / 'dev_qwen_ce32_v1/dev_metrics.json'):
    assert json.loads(path.read_text())['finalized'], f'Finish question-guided retrieval before using {path}'
def run_script(label, filename, arguments):
    script = REPO_DIR / 'scripts' / filename
    if not script.is_file(): raise FileNotFoundError(script)
    old_argv = sys.argv[:]
    sys.argv = [str(script), *map(str, arguments)]
    print(f'\n=== {label} ===', flush=True)
    started = time.monotonic()
    try: runpy.run_path(str(script), run_name='__main__')
    finally:
        sys.argv = old_argv
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f'{label}: {(time.monotonic()-started)/60:.1f} minutes this invocation', flush=True)
READER_ARGS = ['--candidate-k', '20', '--bm25-keep', '8', '--batch-size', '16', '--max-seq-length', '384', '--max-answer-tokens', '24', '--checkpoint-every', '1', '--resume']


## 1. Blind, train-only passage audit

This creates up to 240 unique candidate rows from 40 randomly chosen train claims, without displaying system names or gold answers. Fill `answers_claim`, `decisive_for_verdict`, and `stance` in the CSV (`yes/no/unclear`, `yes/no/unclear`, and `supports/refutes/conflicts/unknown`). Keep the key file private until the audit is complete. The cell never overwrites an existing annotated CSV. The rest of the notebook can run while you annotate.


In [ ]:
import csv, random, hashlib
from src.utils.io import iter_jsonl
audit_dir = ROOT / 'audit'
audit_dir.mkdir(parents=True, exist_ok=True)
audit_csv, audit_key = audit_dir / 'train_passages_blind.csv', audit_dir / 'train_passages_key.json'
if audit_csv.exists() or audit_key.exists():
    if not (audit_csv.exists() and audit_key.exists()): raise RuntimeError('Audit CSV/key pair is incomplete; inspect before regenerating')
    print('Reusing audit files; existing human annotations were not overwritten:', audit_csv)
else:
    qg_by_id = {row['claim_id']: row for row in iter_jsonl(QG_TRAIN)}
    ids = sorted(qg_by_id)
    rng = random.Random(718)
    sampled = set(rng.sample(ids, min(40, len(ids))))
    audit_rows, key = [], {}
    for bm25 in iter_jsonl(TRAIN100):
        claim_id = bm25['claim_id']
        if claim_id not in sampled: continue
        source_items = {item['passage_id']: item for item in bm25['retrieved']}
        appearances = {}
        for system, items in [('BM25', bm25['retrieved'][:3]), ('question_guided', qg_by_id[claim_id]['retrieved'][:3])]:
            for item in items: appearances.setdefault(item['passage_id'], []).append(system)
        for passage_id, systems in appearances.items():
            item = source_items[passage_id]
            audit_id = hashlib.sha256(f'{claim_id}|{passage_id}'.encode()).hexdigest()[:16]
            key[audit_id] = {'claim_id': claim_id, 'passage_id': passage_id, 'systems': systems}
            audit_rows.append({'audit_id': audit_id, 'claim_id': claim_id, 'claim': bm25['claim'], 'passage_id': passage_id, 'url': item.get('url') or '', 'passage_text': item['text'], 'answers_claim': '', 'decisive_for_verdict': '', 'stance': '', 'notes': ''})
    rng.shuffle(audit_rows)
    fields = ['audit_id', 'claim_id', 'claim', 'passage_id', 'url', 'passage_text', 'answers_claim', 'decisive_for_verdict', 'stance', 'notes']
    with audit_csv.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields); writer.writeheader(); writer.writerows(audit_rows)
    audit_key.write_text(json.dumps(key, indent=2), encoding='utf-8')
    print(f'Created {len(audit_rows)} blind rows:', audit_csv)


## 2. Train-only reader smoke test, then full train/dev features

The reader model is extractive and **does not see the annotated answers**. It processes the saved question-guided shortlist plus the first eight BM25 items, at most 20 passages per claim. Progress is per claim; checkpoints are append-only on Drive. The smoke test uses the first 20 **train** claims only. A positive answerability margin is not a correctness guarantee. Reader inference may take hours; rerun the same cell after interruption.


In [ ]:
run_script('Answer reader / train smoke20', 'run_answer_reader.py', [
    '--bm25-rankings', TRAIN100, '--question-checkpoint', QG_TRAIN, '--split', 'train',
    '--output-dir', ROOT / 'reader', '--run-name', 'train_smoke20', '--max-claims', '20', *READER_ARGS,
])
assert (ROOT / 'reader/train_smoke20/train_answer_features.jsonl').is_file()


In [ ]:
run_script('Answer reader / train', 'run_answer_reader.py', [
    '--bm25-rankings', TRAIN100, '--question-checkpoint', QG_TRAIN, '--split', 'train',
    '--output-dir', ROOT / 'reader', '--run-name', 'train_qa20_v1', *READER_ARGS,
])
assert json.loads((TRAIN_READER.parent / 'train_answer_reader_report.json').read_text())['claim_count'] == 3067


In [ ]:
run_script('Answer reader / dev', 'run_answer_reader.py', [
    '--bm25-rankings', DEV100, '--question-checkpoint', QG_DEV, '--split', 'dev',
    '--output-dir', ROOT / 'reader', '--run-name', 'dev_qa20_v1', *READER_ARGS,
])
assert json.loads((DEV_READER.parent / 'dev_answer_reader_report.json').read_text())['claim_count'] == 500


## 3. Train-only silver selector with out-of-fold train rankings

The prior answer-alignment file supplies one *silver* positive passage on some train claims. The selector uses only those train labels, tunes `C` in `{0.1, 1, 10}` on a train-only holdout, and creates five-fold out-of-fold train rankings. Dev qrels are opened **only after** dev rankings are frozen, for the same URL-proxy hit@3/MRR comparison. If too few silver positives survive in the 20-passage shortlist, the script stops rather than pretending it trained reliably.


In [ ]:
run_script('Train answer-aware selector', 'train_answer_selector.py', [
    '--train-features', TRAIN_READER, '--dev-features', DEV_READER, '--train-alignment', ALIGNMENT,
    '--dev-qrels', DEV_QRELS, '--output-dir', ROOT / 'selector', '--run-name', 'qa20_silver_oof_v1',
    '--ranking-k', '20', '--folds', '5', '--min-positive-claims', '30', '--seed', '718',
])
selector_report = json.loads((SELECTOR / 'selector_report.json').read_text())
print('Silver positives in scored candidates:', selector_report['positive_in_scored_candidates_count'])
print('Dev URL-proxy retrieval metrics:', selector_report['dev_url_proxy_metrics'])


## 4. Matched evidence-conditioned verifiers

Both runs use the **same selected passage IDs** for each split; only their text differs. The first is a selection-only control with full source passages. The second supplies short question/candidate-answer/verbatim-source excerpts. DistilBERT hyperparameters are selected using the existing train-only validation procedure, then refit on all official train claims and evaluated once on all 500 dev claims. Rerun either cell after an interruption.


In [ ]:
VERIFIER_ARGS = ['--model', 'distilbert-base-uncased', '--learning-rates', '2e-5,5e-5', '--epoch-grid', '2,3', '--batch-size', '8', '--gradient-accumulation-steps', '2', '--checkpoint-steps', '100', '--seed', '718', '--resume']
run_script('Matched DistilBERT / selected raw passages', 'train_transformer_verifier.py', [
    '--data-root', DATA_ROOT, '--train-rankings', SELECTOR / 'train_raw_rankings.jsonl',
    '--eval-rankings', SELECTOR / 'dev_raw_rankings.jsonl',
    '--output-dir', ROOT / 'verifier', '--run-name', 'answer_aware_raw_v1', *VERIFIER_ARGS,
])


In [ ]:
run_script('Matched DistilBERT / quoted answers', 'train_transformer_verifier.py', [
    '--data-root', DATA_ROOT, '--train-rankings', SELECTOR / 'train_qa_rankings.jsonl',
    '--eval-rankings', SELECTOR / 'dev_qa_rankings.jsonl',
    '--output-dir', ROOT / 'verifier', '--run-name', 'answer_aware_qa_v1', *VERIFIER_ARGS,
])


## 5. Compare against fixed baselines and inspect the audit

The previous dev runs are included as **references**, not reselected after seeing this experiment. Claim-only and gold-answer results are diagnostics, not deployable evidence-grounded systems. Check macro-F1 *and* both error types; higher URL-proxy hit@3 alone does not establish a better fact-checker.


In [ ]:
import pandas as pd
from IPython.display import display
raw = json.loads((ROOT / 'verifier/answer_aware_raw_v1/final_dev_metrics.json').read_text())
qa = json.loads((ROOT / 'verifier/answer_aware_qa_v1/final_dev_metrics.json').read_text())
def row(name, metrics, hit):
    errors = metrics['decision_error_rates']
    result = {'method': name, 'URL-proxy hit@3': hit, 'macro-F1': metrics['macro_f1'], 'accuracy': metrics['accuracy'], 'false reassurance /500': errors['false_reassurance_rate'], 'false alarm /500': errors['false_alarm_rate']}
    for label in ('Supported', 'Refuted', 'Not Enough Evidence', 'Conflicting Evidence'):
        result[label + ' F1'] = metrics['per_class'][label]['f1-score']
    return result
hit = selector_report['dev_url_proxy_metrics']['evidence_hit_at_3']
rows = [
    {'method': 'BM25 + Qwen3-4B (prior)', 'URL-proxy hit@3': .196891, 'macro-F1': .3346, 'accuracy': .496, 'false reassurance /500': .220, 'false alarm /500': .058},
    {'method': 'BM25 + DistilBERT (prior)', 'URL-proxy hit@3': .196891, 'macro-F1': .346929, 'accuracy': .520, 'false reassurance /500': .148, 'false alarm /500': .156},
    {'method': 'BGE-reranked BM25 + DistilBERT (exploratory prior)', 'URL-proxy hit@3': .186528, 'macro-F1': .385443, 'accuracy': .552, 'false reassurance /500': .176, 'false alarm /500': .136},
    {'method': 'Question-guided excerpts + DistilBERT (prior)', 'URL-proxy hit@3': .233161, 'macro-F1': .359721, 'accuracy': .528, 'false reassurance /500': .176, 'false alarm /500': .136},
    {'method': 'Claim-only DistilBERT (diagnostic)', 'URL-proxy hit@3': None, 'macro-F1': .396978, 'accuracy': .576, 'false reassurance /500': .136, 'false alarm /500': .136},
    {'method': 'Gold-answer DistilBERT (oracle diagnostic)', 'URL-proxy hit@3': None, 'macro-F1': .528395, 'accuracy': .636, 'false reassurance /500': .116, 'false alarm /500': .088},
    row('Answer-aware + raw passages', raw, hit), row('Answer-aware + quoted answers', qa, hit),
]
display(pd.DataFrame(rows).round(4))
with audit_csv.open(encoding='utf-8', newline='') as handle: audit = list(csv.DictReader(handle))
done = [item for item in audit if item['answers_claim'].strip() and item['decisive_for_verdict'].strip()]
print(f'Manual audit completed: {len(done)}/{len(audit)} rows')
if done:
    key = json.loads(audit_key.read_text())
    for system in ('BM25', 'question_guided'):
        selected = [item for item in done if system in key[item['audit_id']]['systems']]
        print(system, 'annotated passages answering claim:', sum(item['answers_claim'].strip().lower() == 'yes' for item in selected), '/', len(selected), '; decisive:', sum(item['decisive_for_verdict'].strip().lower() == 'yes' for item in selected), '/', len(selected))
    if len(done) == len(audit):
        claims = sorted({item['claim_id'] for item in audit})
        for system in ('BM25', 'question_guided'):
            hit = sum(any(item['claim_id'] == claim_id and system in key[item['audit_id']]['systems'] and item['decisive_for_verdict'].strip().lower() == 'yes' for item in audit) for claim_id in claims)
            print(system, 'human decisive-evidence hit@3:', hit, '/', len(claims))


### Interpretation rules

1. The answer-aware selector cannot recover a source absent from its **20 scored candidates**, even if BM25 contains it at positions 21–100. Report that candidate ceiling separately from achieved hit@3.
2. The silver alignment is incomplete and noisy; never describe a high silver-validation score as human passage relevance. The manual audit remains the stronger check.
3. The QA reader may select a confidently wrong span. Its output says *candidate answer* and preserves a verbatim source excerpt; inspect citations before using it in an interface.
4. The same official dev split has already guided several exploratory experiments. Treat new dev gains as exploratory, not an unbiased final model-selection result. Do not feed dev gold answers, verdicts, or qrels into retrieval/reader/selector training.
5. Foundation models can retain knowledge of historical claims. Reusing Qwen-generated questions does not remove that possible pretraining contamination.
